## Setup do Ambiente
- Importação de bibliotecas essenciais do PySpark
- Definição dos paths utilizados para as tabelas
- Utilização do catálogo `catalogo`, schemas `silver_db_name`, `gold_db_name`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    explode,
    sequence,
    col,
    year,
    quarter,
    month,
    weekofyear,
    dayofmonth,
    dayofweek,
    when
)
from pyspark.sql.types import DateType
from functools import reduce

### Definição de Variáveis Globais
- Centralizamos os nomes de catálogos, bancos de dados e caminhos.
- **Boas Práticas:** Evitar "hardcoding" (escrever o caminho diretamente no código várias vezes). Se o nome do catálogo mudar no futuro, alteramos apenas aqui.

In [0]:
catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
gold_db_name = "gold_credit"

In [0]:
spark.sql(f"USE CATALOG {catalogo};")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_db_name};")
spark.sql(f"USE SCHEMA {gold_db_name};")

## Funções Úteis

### Função `table_check`

Esta função verifica se uma tabela existe em um banco de dados específico e se ela contém dados.

In [0]:
def table_check(table_name, db_name):
    """
    Verifica se uma tabela existe e possui dados em um banco de dados especificado.

    Args:
        table_name (str): Nome da tabela a ser verificada.
        db_name (str): Nome do banco de dados onde a tabela está localizada.

    Returns:
        bool: True se a tabela existe e possui dados, False caso contrário.

    Raises:
        ValueError: Se a tabela existe mas está vazia.
    """
    if spark.catalog.tableExists(f"{db_name}.{table_name}"):
        if spark.table(f"{db_name}.{table_name}").count() == 0:
            raise ValueError(f"Tabela {db_name}.{table_name} existe mas está vazia.")
        return True
    return False

### Função `save_table_gold` 
Salva uma tabela a partir do catálogo (`catalogo`) e camada (`gold_db_name`) especificados, em formato Delta.
Adiciona a coluna `data_criacao_gold`

In [0]:
def save_table_gold(table_name: str, df, process_col=True):
    """
    Salva um DataFrame como tabela Delta na camada gold.

    Args:
        table_name (str): Nome da tabela a ser criada ou sobrescrita na camada gold.
        df (DataFrame): DataFrame Spark que será salvo como tabela Delta.
        process_col(Bool): Adicionar uma coluna com o current timestamp.
    """
    table_path = f"{catalogo}.{gold_db_name}.{table_name}"

    try:
        old_schema = spark.table(table_path).schema.simpleString()
    except AnalysisException:
        old_schema = None

    if process_col == True:
        df = df.withColumn("data_criacao_gold", current_timestamp())
    
    try:
        df.write \
            .format("delta") \
            .option("overwriteSchema", "true") \
            .mode("overwrite") \
            .saveAsTable(table_path)
    except Exception as ex:
        print(f"Erro ao salvar a tabela {table_path}: {ex}")
        return
    
    new_schema = spark.table(table_path).schema.simpleString()

    if old_schema is None:
        print(f"Tabela {table_path} criada pela primeira vez.")
    elif old_schema != new_schema:
        print(f"Esquema da tabela {table_path} foi alterado.\n")
        print("Schema anterior:")
        print(old_schema)
        print("\nNovo schema:")
        print(new_schema)
    else:
        print(f"Tabela salva com sucesso: {table_path}")

### Função `read_table`
Lê uma tabela do Databricks a partir do catálogo e camada especificados (`silver` ou `gold`), retornando um DataFrame Spark correspondente.

In [0]:
def read_table(nome_tabela: str, camada:str='silver'):
    """
    Lê uma tabela Delta da camada especificada.

    Args:
        nome_tabela (str): Nome da tabela a ser lida.
        camada (str, optional): Camada de origem da tabela ('silver' ou 'gold'). Default é 'bronze'.

    Returns:
        DataFrame: DataFrame Spark da tabela lida.

    Raises:
        ValueError: Se a camada não for 'silver' ou 'gold'.
        ValueError: Se a tabela não existir ou estiver vazia.
    """
    db_map = {
        'silver': silver_db_name,
        'gold': gold_db_name
    }

    db_nome = db_map.get(camada.lower())
    
    if not db_nome:
        raise ValueError("Camada deve ser 'silver' ou 'gold'")
    
    if not table_check(nome_tabela, db_nome):
        raise ValueError(f"Tabela {db_nome}.{nome_tabela} não existe.")
    return spark.table(f"{catalogo}.{db_nome}.{nome_tabela}")

## Criação da tabela `dm_tempo`

Colunas da tabela `dm_tempo`:

- `sk_tempo`
- `ano`
- `trimestre`
- `mes`
- `semana_do_ano`
- `dia`
- `dia_da_semana_num`
- `dia_da_semana_nome`
- `mes_nome`
- `eh_fim_de_semana`

In [0]:
data_inicio = '2016-01-01'
data_fim = '2019-01-01'

df_datas = (
    spark.createDataFrame([(data_inicio, data_fim)], ['data_inicio', 'data_fim'])
    .select(explode(sequence(col('data_inicio').cast(DateType()), col('data_fim').cast(DateType()))).alias('sk_tempo'))
    .withColumn('ano', year(col('sk_tempo')))
    .withColumn('trimestre', quarter(col('sk_tempo')))
    .withColumn('mes', month(col('sk_tempo')))
    .withColumn('semana_do_ano', weekofyear(col('sk_tempo')))
    .withColumn('dia', dayofmonth(col('sk_tempo')))
    .withColumn('dia_da_semana_num', dayofweek(col('sk_tempo')))
    .withColumn('dia_da_semana_nome', 
        when(col('dia_da_semana_num') == 1, 'Domingo')
        .when(col('dia_da_semana_num') == 2, 'Segunda-feira')
        .when(col('dia_da_semana_num') == 3, 'Terça-feira')
        .when(col('dia_da_semana_num') == 4, 'Quarta-feira')
        .when(col('dia_da_semana_num') == 5, 'Quinta-feira')
        .when(col('dia_da_semana_num') == 6, 'Sexta-feira')
        .when(col('dia_da_semana_num') == 7, 'Sabado')
    )
    .withColumn('mes_nome',
        when(col('mes') == 1, 'Janeiro')
        .when(col('mes') == 2, 'Fevereiro')
        .when(col('mes') == 3, 'Março')
        .when(col('mes') == 4, 'Abril')
        .when(col('mes') == 5, 'Maio')
        .when(col('mes') == 6, 'Junho')
        .when(col('mes') == 7, 'Julho')
        .when(col('mes') == 8, 'Agosto')
        .when(col('mes') == 9, 'Setembro')
        .when(col('mes') == 10, 'Outubro')
        .when(col('mes') == 11, 'Novembro')
        .when(col('mes') == 12, 'Dezembro')
    )
    .withColumn('eh_fim_de_semana', when(col('dia_da_semana_num').isin([1,7]), 'Sim').otherwise('Não'))
)

df_datas.limit(20).display()

df_datas.write.mode('overwrite').saveAsTable(f'{catalogo}.{gold_db_name}.dm_tempo')

## Criação da tabela `ft_chamados`

Colunas da tabela `ft_chamados`:

- `id_chamado`
- `id_cliente`
- `id_atendente`
- `motivo`
- `canal`
- `resolvido`
- `nota_atendimento`
- `categoria_nota`
- `status_canal`
- `valor_custo`

In [0]:
df_chamados_geral = spark.table(f'{catalogo}.{silver_db_name}.ft_chamados_geral')
display(df_chamados_geral.limit(20))

In [0]:
print(f"Colunas de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
print(f"{df_chamados_geral.columns}\n")
print(f"Schema de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
df_chamados_geral.printSchema()

In [0]:
df_gold_ft_chamados = df_chamados_geral.select(
    col('id_chamado'), col('id_cliente'),     
    when(col('id_atendente') == -1, None).otherwise(col('id_atendente')).alias('id_atendente'), 
    col('motivo'), col('canal'), col('status_canal'), col('resolvido'), col('nota_atendimento'), col('categoria_nota'), col('valor_custo')
)
df_gold_ft_chamados.limit(15).display()
df_gold_ft_chamados.printSchema()

In [0]:
df_gold_ft_chamados.write.mode('overwrite').saveAsTable(f'{catalogo}.{gold_db_name}.ft_chamados')